# ML-07 — Baseline Action Score and Top-10 Review

Simple words, honest numbers. Data: the starter CSV in this repo (`data/raw/content_refresh_anonymized.csv`, one row per content item).

**Lane check:** I am confirming **Lane 2, Refresh / Content Opportunity Scoring**. Nothing in the two signal checks below argues for switching.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**A word on the label.** There is no future outcome in this file, so I judge signals and the rule against a *proxy*: `is_declining` = (`trend_direction == "down"`), meaning impressions fell more than 20% in the last 30 days against the 30 before. That proxy is used **only to check my work**. It is never an input to the rule, and neither are `trend_direction`, `trend_pct` or any `*_last_30d` / `*_prev_30d` column.

In [1]:
import json, os
import numpy as np
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)   # evaluation only, never an input

BASE_RATE = df["is_declining"].mean()
print(f"rows: {len(df):,} | base rate of the declining proxy: {BASE_RATE:.3f}")

rows: 30,000 | base rate of the declining proxy: 0.542


### Signal check 1 — staleness (the idea behind FlyRank's refresh flags)

Claim I want to lean on: pages that have gone longer without an update are more likely to be sliding.

In [2]:
bins   = [0, 30, 90, 180, np.inf]
labels = ["0-29 days", "30-89 days", "90-179 days", "180+ days"]
df["stale_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, right=False)

sig1 = (df.groupby("stale_bucket", observed=True)["is_declining"]
          .agg(n="size", declining_rate="mean").round(3))
sig1

,n,declining_rate
stale_bucket,,
0-29 days,20480,0.511
30-89 days,175,0.589
90-179 days,9171,0.611
180+ days,174,0.471


**Verdict 1: MIXED.** The pattern holds from fresh pages (51% declining, n=20,480) to pages 90 to 179 days old (61%, n=9,171), which is about ten points higher. It does not keep going: the 180+ bucket is lower (47%), but it holds only 174 pages, so I can't read much into it. I'll use staleness as a *moderate* signal, only inside the 90+ range where the table has real counts.

### Signal check 2 — CTR against position (the idea behind FlyRank's CTR-fix logic)

Claim: a page that already ranks on page one and is visible, but gets a very low click-through rate, is more likely to be sliding. I limit this to pages with at least 500 impressions and a real position (0 means "no data").

In [3]:
vis_p1 = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 10)].copy()
vis_p1["ctr_quartile"] = pd.qcut(vis_p1["ctr"], 4, duplicates="drop")

sig2 = (vis_p1.groupby("ctr_quartile", observed=True)["is_declining"]
          .agg(n="size", declining_rate="mean").round(3))
print("CTR is a percentage here (0.12 means 0.12%).")
sig2

CTR is a percentage here (0.12 means 0.12%).


,n,declining_rate
ctr_quartile,,
"(-0.001, 0.12]",2029,0.715
"(0.12, 0.24]",1872,0.599
"(0.24, 0.44]",1779,0.561
"(0.44, 5.42]",1884,0.469


**Verdict 2: CONFIRMED.** Among visible page-one pages, the declining rate falls steadily as CTR rises: about 72% for the lowest-CTR quartile down to about 47% for the highest (every bucket has n above 1,700). The lowest quartile ends at a CTR of 0.12%, so I use 0.12% as my cutoff, read off this table and not searched for.

*Caveat:* this is an association inside one 90-day snapshot. It shows that low-CTR pages tend to be the ones falling now, not that fixing the title will stop the fall.

### The rule, in plain words

A page is worth an editor's time if it is **visible** (at least 500 impressions in 90 days) and either (a) it sits on page one but gets a **very low CTR** (0.12% or less), or (b) it has gone **90 or more days without an update**. Evidence points: CTR gap = 2, stale = 1, so a page with both scores 3.

**Reason codes (one per row):** `ctr_gap_and_stale` (3 points), `ctr_gap_page1` (2), `stale_but_visible` (1), `no_flag` (0).
Ties are broken by evidence, not by size: lowest CTR first, then longest since the last update.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
VISIBLE   = df["impressions_90d"] >= 500
PAGE_ONE  = (df["avg_position"] > 0) & (df["avg_position"] <= 10)
CTR_GAP   = VISIBLE & PAGE_ONE & (df["ctr"] <= 0.12)
STALE     = VISIBLE & (df["days_since_last_update"] >= 90)

df["score"] = 2 * CTR_GAP.astype(int) + STALE.astype(int)

df["reason_code"] = np.select(
    [CTR_GAP & STALE, CTR_GAP, STALE],
    ["ctr_gap_and_stale", "ctr_gap_page1", "stale_but_visible"],
    default="no_flag",
)
df["action_label"] = df["reason_code"].map({
    "ctr_gap_and_stale": "Refresh content and rewrite title/meta for CTR",
    "ctr_gap_page1":     "Rewrite title/meta for CTR",
    "stale_but_visible": "Refresh content",
    "no_flag":           "No action",
})

# Inputs the rule is allowed to read. Nothing else feeds the score.
RULE_INPUTS = ["impressions_90d", "avg_position", "ctr", "days_since_last_update"]

queue = (df.sort_values(["score", "ctr", "days_since_last_update"], ascending=[False, True, False], kind="stable")
           .reset_index(drop=True))
queue.insert(0, "rank", queue.index + 1)

os.makedirs("../outputs", exist_ok=True)
out_cols = ["rank", "content_id", "client_id", "score", "reason_code", "action_label"] + RULE_INPUTS
queue[out_cols].to_csv("../outputs/baseline_action_score.csv", index=False)

print("rows written to work/outputs/baseline_action_score.csv:", len(queue))
print(queue["reason_code"].value_counts().to_string())

rows written to work/outputs/baseline_action_score.csv: 30000
reason_code
no_flag              22245
stale_but_visible     5726
ctr_gap_page1         1180
ctr_gap_and_stale      849


**Does the ranking beat random?** Precision@K is the share of the top K that are in the declining proxy, shown next to the base rate. This check uses the label, which is fine because it is evaluation, not input.

In [5]:
def precision_at_k(frame, k):
    return frame["is_declining"].head(k).mean()

ks = [10, 20, 50, 100, 500]
prec = {k: round(precision_at_k(queue, k), 3) for k in ks}
print(f"base rate (random picking): {BASE_RATE:.3f}")
print("precision@K, evidence-ranked:", prec)

by_code = (queue.groupby("reason_code")["is_declining"].agg(n="size", declining_rate="mean").round(3))
by_code

base rate (random picking): 0.542
precision@K, evidence-ranked: {10: np.float64(0.7), 20: np.float64(0.7), 50: np.float64(0.78), 100: np.float64(0.86), 500: np.float64(0.764)}


,n,declining_rate
reason_code,,
ctr_gap_and_stale,849,0.749
ctr_gap_page1,1180,0.691
no_flag,22245,0.512
stale_but_visible,5726,0.597


**What I tried first, and why I changed it.** My first score weighted every flagged page by its impressions, so the biggest pages came first. The cell below reproduces that. It did *worse than random* at the top. Large pages decline less often here, which matches the mixed picture in the impression tiers. So I switched the tie-break to evidence strength. Note that I made that change **after** seeing precision@K, so the final numbers above are a little optimistic.

In [6]:
first_try = df.assign(first_score=df["impressions_90d"] * df["score"]) \
              .sort_values("first_score", ascending=False, kind="stable")
first_prec = {k: round(first_try["is_declining"].head(k).mean(), 3) for k in [10, 50, 100]}
print(f"first attempt (impressions x points) precision@K: {first_prec}   vs base rate {BASE_RATE:.3f}")

first attempt (impressions x points) precision@K: {10: np.float64(0.3), 50: np.float64(0.48), 100: np.float64(0.4)}   vs base rate 0.542


In [7]:
metrics = {
    "task": "w04_baseline_score",
    "n_rows": int(len(df)),
    "label_proxy": "trend_direction == down (evaluation only)",
    "base_rate": round(float(BASE_RATE), 4),
    "signal_1": {"name": "staleness", "verdict": "MIXED"},
    "signal_2": {"name": "ctr_vs_position", "verdict": "CONFIRMED"},
    "n_flagged": int((df["score"] > 0).sum()),
    "precision_at_k": {str(k): float(v) for k, v in prec.items()},
    "first_attempt_precision_at_k": {str(k): float(v) for k, v in first_prec.items()},
    "rule_inputs": RULE_INPUTS,
}
with open("../outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

{
  "task": "w04_baseline_score",
  "n_rows": 30000,
  "label_proxy": "trend_direction == down (evaluation only)",
  "base_rate": 0.5421,
  "signal_1": {
    "name": "staleness",
    "verdict": "MIXED"
  },
  "signal_2": {
    "name": "ctr_vs_position",
    "verdict": "CONFIRMED"
  },
  "n_flagged": 7755,
  "precision_at_k": {
    "10": 0.7,
    "20": 0.7,
    "50": 0.78,
    "100": 0.86,
    "500": 0.764
  },
  "first_attempt_precision_at_k": {
    "10": 0.3,
    "50": 0.48,
    "100": 0.4
  },
  "rule_inputs": [
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update"
  ]
}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

*One line each: the action, why it is there, and what would make it wrong. The lines below are drafted from each row's own values. Read every one and edit it until you would say it yourself.*

In [8]:
def why(r):
    parts = [f"{r.impressions_90d:,} impressions in 90 days", f"average position {r.avg_position}", f"CTR {r.ctr}%"]
    if r.score >= 1 and r.days_since_last_update >= 90:
        parts.append(f"last updated {r.days_since_last_update} days ago")
    return ", ".join(parts)

def wrong(r):
    intent = r.main_intent if isinstance(r.main_intent, str) else "unknown"
    if r.reason_code == "stale_but_visible":
        return "Wrong if the page is evergreen and its content hasn't aged, so an update would change nothing."
    base = ("Wrong if the query rarely produces a click, for example when the answer shows in the search result itself")
    if intent == "navigational":
        return base + "; the intent is navigational, so people may not need to click at all."
    if intent == "unknown":
        return base + "; the intent is unknown, so I can't tell."
    return base + f"; the intent is {intent}. Also wrong if the low CTR is a measurement gap and not a real click problem."

top10 = queue.head(10).copy()
top10["page"]  = top10["content_id"].str[-6:]
top10["why"]   = top10.apply(why, axis=1)
top10["what would make it wrong"] = top10.apply(wrong, axis=1)

review = top10[["rank", "page", "action_label", "reason_code", "why", "what would make it wrong"]]
pd.set_option("display.max_colwidth", 140)
review.to_string(index=False)

' rank   page                                   action_label       reason_code                                                                                     why                                                                                                                                                                                             what would make it wrong\n    1 2484db Refresh content and rewrite title/meta for CTR ctr_gap_and_stale 1,149 impressions in 90 days, average position 4.1, CTR 0.0%, last updated 106 days ago Wrong if the query rarely produces a click, for example when the answer shows in the search result itself; the intent is informational. Also wrong if the low CTR is a measurement gap and not a real click problem.\n    2 31ab45 Refresh content and rewrite title/meta for CTR ctr_gap_and_stale   595 impressions in 90 days, average position 5.9, CTR 0.0%, last updated 106 days ago    Wrong if the query rarely produces a click, for example when the answe

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# Weak picks: use the label here only to judge, never to score.
top50 = queue.head(50)
print(f"top 50: {int(top50['is_declining'].sum())} of 50 in the declining proxy ({top50['is_declining'].mean():.0%}); base rate {BASE_RATE:.0%}")

print("\ntop-10 picks that are NOT in the declining proxy:")
weak = queue.head(10).query("is_declining == 0")[["rank", "reason_code", "trend_direction"]]
print(weak.to_string(index=False))

print("\nhow concentrated is the top 100 by client (pseudonymous ids, last 4 chars)?")
conc = queue.head(100)["client_id"].str[-4:].value_counts().head(5)
print(conc.to_string())

print("\nhow bunched are the top 10 by days since update?")
print(queue.head(10)["days_since_last_update"].describe()[["min", "max"]].to_string())

top 50: 39 of 50 in the declining proxy (78%); base rate 54%

top-10 picks that are NOT in the declining proxy:
 rank       reason_code trend_direction
    3 ctr_gap_and_stale              up
    7 ctr_gap_and_stale          stable
   10 ctr_gap_and_stale          stable

how concentrated is the top 100 by client (pseudonymous ids, last 4 chars)?


client_id
27de    49
0f77    23
5f04    11
8562    11
d7e2     3

how bunched are the top 10 by days since update?
min    104.0
max    106.0


In [10]:
# Leakage guard: the rule may read only RULE_INPUTS, and none of them may be label-derived or a recent-window column.
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining"} | {c for c in df.columns if c.endswith("_last_30d") or c.endswith("_prev_30d")}
assert not (set(RULE_INPUTS) & FORBIDDEN), "a label-derived or recent-window column leaked into the rule"
print("leakage guard passed. rule inputs:", RULE_INPUTS)

leakage guard passed. rule inputs: ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update']


**Weak picks and limits (from the output above):**

- *Three of my top 10 look wrong against the proxy:* rank 3 is trending **up** and ranks 7 and 10 are **stable**. So the top 10 are right 7 times out of 10, and the rule is honestly beatable.
- *One client dominates:* a single client supplies 49 of the top 100, and the top three clients supply 83. The ranking may be surfacing that client's habits (for example a batch update), not a general pattern. Week 5 should test on clients the rule hasn't seen.
- *The top 10 are bunched:* every one has a CTR of 0.0% and was last updated 104 to 106 days ago. That also points to a batch of pages updated together.
- *Overlap:* `impressions_90d` and `ctr` cover 90 days, which includes the last 30 days the label is built from. The starter pipeline uses the same window logic, but it is a partial overlap, and I name it as a limit.
- *The proxy:* "declining" is not the same as "worth a refresh". A page can decline for reasons an editor can't fix.
- *The first attempt:* weighting by impressions put 30% right in the top 10, below the 54% base rate, so volume is not a safe weight here.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.